In [1]:
import sys, os, importlib
package_path = os.path.abspath('../../')
if package_path not in sys.path:
    sys.path.append(package_path)
from package import functions as fn
from package import model as rm
from package import optimizer as opt
from package import plots
import numpy as np
import pandas as pd
import nbformat as nbf
from obspy.taup import TauPyModel
from obspy.geodetics import gps2dist_azimuth
from obspy.geodetics import locations2degrees, kilometers2degrees
from obspy import read
from matplotlib import pyplot as plt

np.set_printoptions(precision=4, suppress=True)
for module in [fn, rm, opt, plots]:
    importlib.reload(module)

In [2]:
# see all paths in sys
for p in sys.path:
    print(p)

/home/vay3059/desktop/SURG-Mars/project/tests/actual
/opt/anaconda3/lib/python312.zip
/opt/anaconda3/lib/python3.12
/opt/anaconda3/lib/python3.12/lib-dynload

/home/vay3059/.local/lib/python3.12/site-packages
/opt/anaconda3/lib/python3.12/site-packages
/home/vay3059/desktop/SURG-Mars/project


In [3]:
test_data_path = '../../data/test_data.csv'
data = pd.read_csv(test_data_path)
data

,Item,Eventid,Magnitude,Origin_time,Longitude,Latitude,Depth,Station,Loc_Long,Loc_Lat,...,ELPSZ,Band_Pass_P,EBPP,Band_Pass_SV,EBPSV,Band_Pass_SH,EBPSH,Band_Pass_SZ,EBPSZ,Website
0,1,usp000ahq6,5.3,6/29/2001 23:40,29.972,0.292,10.0,MBAR,30.74,-0.60,...,65900.0,425000.0,552.00,385000.0,195000.0,-2160000.0,114000.0,1140000.0,127000.0,https://service.iris.edu/irisws/traveltime/1/q...
1,2,usp000c479,5.2,8/5/2003 18:56,29.446,-0.521,10.0,KMBO,37.25,-1.13,...,420.0,-13.3,5.05,-5860.0,1180.0,4940.0,621.0,-3560.0,589.0,https://service.iris.edu/irisws/traveltime/1/q...
2,3,usp000efkj,5.2,4/27/2006 4:18,30.078,0.338,10.0,MBAR,30.74,-0.60,...,670.0,-1610.0,956.00,5380.0,1900.0,-5130.0,2100.0,9230.0,2540.0,https://service.iris.edu/irisws/traveltime/1/q...
3,4,usp000ej88,4.9,5/29/2006 15:30,30.114,0.343,23.9,MBAR,30.74,-0.60,...,598.0,5670.0,2110.00,11700.0,4390.0,8600.0,5020.0,-5580.0,4350.0,https://service.iris.edu/irisws/traveltime/1/q...
4,5,usp000f54h,5.6,2/19/2007 2:33,30.758,1.750,19.0,KMBO,37.25,-1.13,...,1300.0,-168.0,5.15,-9720.0,1900.0,7710.0,1320.0,-4400.0,1360.0,https://service.iris.edu/irisws/traveltime/1/q...
5,6,usp000fe2y,5.9,6/15/2007 18:49,30.834,1.719,24.0,KMBO,37.25,-1.13,...,2450.0,-503.0,12.40,16400.0,3780.0,-13500.0,2440.0,-12500.0,2330.0,https://service.iris.edu/irisws/traveltime/1/q...
6,7,usp000gjbn,5.3,10/5/2008 0:02,29.118,-1.126,4.0,KMBO,37.25,-1.13,...,405.0,95.4,7.41,-1280.0,540.0,2200.0,378.0,1780.0,414.0,https://service.iris.edu/irisws/traveltime/1/q...
7,8,usp000h343,4.9,10/18/2009 0:39,30.145,0.563,10.0,KMBO,37.25,-1.13,...,1010.0,-102.0,5.93,1500.0,340.0,-1340.0,244.0,1090.0,232.0,https://service.iris.edu/irisws/traveltime/1/q...
8,9,usp000h6r2,4.9,1/28/2010 23:52,29.198,-0.901,10.0,KMBO,37.25,-1.13,...,178.0,82.6,7.03,-779.0,356.0,983.0,189.0,294.0,267.0,https://service.iris.edu/irisws/traveltime/1/q...
9,10,usp000hqv8,4.9,12/12/2010 20:34,29.670,0.804,10.0,KMBO,37.25,-1.13,...,13600.0,14100.0,135.00,-60900.0,8580.0,-76000.0,12300.0,16300.0,6620.0,https://service.iris.edu/irisws/traveltime/1/q...


In [4]:
events = 'Eventid'

for name in data[events]:
    nb = nbf.v4.new_notebook()
    if not os.path.exists(f'{name}.ipynb'):
        with open(f'{name}.ipynb', 'w') as f:
            nbf.write(nb, f)
    else: print(f'{name}.ipynb already exists')

usp000ahq6.ipynb already exists
usp000c479.ipynb already exists
usp000efkj.ipynb already exists
usp000ej88.ipynb already exists
usp000f54h.ipynb already exists
usp000fe2y.ipynb already exists
usp000gjbn.ipynb already exists
usp000h343.ipynb already exists
usp000h6r2.ipynb already exists
usp000hqv8.ipynb already exists


### RUN SOME CHECKS ON THE DATA

#### Incidence Angle, 2 Different Ways

In [5]:
# collect all eq and station coords
epidistances = []
azimuths = []
back_azimuths = []

# now get p and s wave velocities
velocity_model = TauPyModel(model='ak135')
vel_model_path = '../../data/AK135_lookup.csv'
lookup_table = pd.read_csv(vel_model_path)

for event in data.itertuples():
    # epdist
    print(f"Event ID: {event.Eventid}")
    eq_lat = event.Latitude
    eq_long = event.Longitude
    sta_lat = event.Loc_Lat
    sta_long = event.Loc_Long
    event_locs = [eq_lat, eq_long, sta_lat, sta_long]
    dist, az, baz = gps2dist_azimuth(*event_locs)
    epdist = kilometers2degrees(dist/1000)
    epidistances.append(epdist)
    azimuths.append(az)
    back_azimuths.append(baz)
    
    # angle check
    hdepth = event.Depth
    alpha, beta = fn.extract_velocities(lookup_table, hdepth)
    velocities = [alpha, beta]
    s_critical = np.rad2deg(np.arcsin(beta/alpha))
    print(f'critical angle: {s_critical: .2f} degrees')
    
    # do the incidence angle check
    p_arrival = velocity_model.get_travel_times(source_depth_in_km=hdepth,
                        distance_in_degree=epdist, phase_list=['P'])[0]
    s_arrival = velocity_model.get_travel_times(source_depth_in_km=hdepth,
                        distance_in_degree=epdist, phase_list=['S'])[0]
    p_inc = p_arrival.incident_angle
    s_inc = s_arrival.incident_angle
    comparison = '>'
    if s_inc <= s_critical: comparison = '<='
    print(f"j = {s_inc: .2f}, j_c = {s_critical: .2f}. j {comparison} j_c")
    
    # second check:
    lp_p_z = event.Low_Pass_P
    lp_sv_r = event.Low_Pass_SV
    lp_sh_t = event.Low_Pass_SH
    lp_sz_z = event.Low_Pass_SZ
    
    # check that things are fine
    sv_on_q = np.sqrt(lp_sv_r**2 + lp_sz_z**2)
    sv_thru_j = lp_sv_r/np.cos(np.deg2rad(s_inc))
    print(f"SV1 = {sv_on_q: .2f}, SV2 = {sv_thru_j: .2f}")
    print(f"Rel error = {100*(sv_thru_j - sv_on_q)/sv_on_q: .2f}%\n")
    

Event ID: usp000ahq6
critical angle:  36.62 degrees
j =  63.62, j_c =  36.62. j > j_c
SV1 =  589665.16, SV2 =  1098158.17
Rel error =  86.23%

Event ID: usp000c479
critical angle:  36.62 degrees
j =  49.95, j_c =  36.62. j > j_c
SV1 =  6513.65, SV2 = -9169.36
Rel error = -240.77%

Event ID: usp000efkj
critical angle:  36.62 degrees
j =  63.62, j_c =  36.62. j > j_c
SV1 =  10460.07, SV2 =  12759.45
Rel error =  21.98%

Event ID: usp000ej88
critical angle:  36.32 degrees
j =  50.18, j_c =  36.32. j > j_c
SV1 =  12168.50, SV2 =  16396.30
Rel error =  34.74%

Event ID: usp000f54h
critical angle:  36.62 degrees
j =  49.99, j_c =  36.62. j > j_c
SV1 =  11922.35, SV2 =  16798.35
Rel error =  40.90%

Event ID: usp000fe2y
critical angle:  36.32 degrees
j =  49.99, j_c =  36.32. j > j_c
SV1 =  26050.91, SV2 =  34841.53
Rel error =  33.74%

Event ID: usp000gjbn
critical angle:  36.62 degrees
j =  49.93, j_c =  36.62. j > j_c
SV1 =  3632.99, SV2 =  4893.89
Rel error =  34.71%

Event ID: usp000h343

In [6]:
new_data_path = '../../data/2025-07-20-mww74-off-east-coast-of-kamchatka-6/II.ERM.00.BH1.M.2025.201.064746.SAC'
st = read(new_data_path)
st[0].stats
    
# # plot only the lh
# for tr in st:
#     if tr.stats.channel.startswith('LH'):
#         print(tr.stats)
#         tr.plot()


         network: II
         station: ERM
        location: 00
         channel: BH1
       starttime: 2025-07-20T06:47:46.019538Z
         endtime: 2025-07-20T06:57:42.969538Z
   sampling_rate: 20.0
           delta: 0.05
            npts: 11940
           calib: 4013060000.0
         _format: SAC
             sac: AttribDict({'delta': 0.05, 'scale': 4013060000.0, 'b': 0.000538, 'e': 596.95056, 'o': 77.98046, 'stla': 42.015, 'stlo': 143.1572, 'stel': 40.0, 'stdp': 0.0, 'evla': 52.8271, 'evlo': 160.7907, 'evdp': 34.0, 'dist': 1785.495, 'az': 234.69592, 'baz': 41.646065, 'gcarc': 16.058054, 'cmpaz': 359.9, 'cmpinc': 90.0, 'nzyear': 2025, 'nzjday': 201, 'nzhour': 6, 'nzmin': 47, 'nzsec': 46, 'nzmsec': 19, 'nvhdr': 6, 'npts': 11940, 'iftype': 1, 'leven': 1, 'kstnm': 'ERM', 'kevnm': 'Off EastCoast O', 'khole': '00', 'kcmpnm': 'BH1', 'knetwk': 'II', 'kinst': 'Metrozet'})

#### STEPS FOR PROCESSING SEISMOGRAMS

* Look under low pass
* Convert from 12 to RT
* [Using ObsPy for the whole seismogram]
- Convert from ZR to Lp and Qp, keep Lp
- Covert from ZR to Ls and Qs, keep Qs

* S-wave Noise on Q and T seismograms should be similar
* Save amplitude plots like in Maddy's paper

Note: I need to extract one good seismogram from IRIS.
I already did, now I need to get it from the data folder...